# ETL and Data Cleaning

## Objectives

This notebook will:

- extract the raw online retail transaction data;
- inspect the dataset structure and data types;
- assess missing values and duplicated records;
- identify cancellations, returns, and invalid values;
- document data-quality limitations;
- transform the data without changing the original CSV;
- save a cleaned dataset for later analysis.

The original file stored in `data/raw` will remain unchanged.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:,.2f}".format)

## Extract the raw dataset

The project uses `pathlib` to construct a reliable path to the raw dataset. The code works when the notebook is opened from either the project root or the `jupyter_notebooks` directory.

In [2]:
project_root = Path.cwd()

if project_root.name == "jupyter_notebooks":
    project_root = project_root.parent

data_path = project_root / "data" / "raw" / "online_retail.csv"

if not data_path.exists():
    raise FileNotFoundError(f"Dataset not found: {data_path}")

print(f"Project root: {project_root}")
print(f"Dataset: {data_path}")

Project root: /Users/ewa/Documents/vscode-projects/code-institute-2026/CI-Project2-Online Retail Transaction Analysis/CI-DA-Project-2-Online-Retail-Transaction-Analysis
Dataset: /Users/ewa/Documents/vscode-projects/code-institute-2026/CI-Project2-Online Retail Transaction Analysis/CI-DA-Project-2-Online-Retail-Transaction-Analysis/data/raw/online_retail.csv


In [3]:
df_raw = pd.read_csv(
    data_path,
    dtype={
        "InvoiceNo": "string",
        "StockCode": "string",
        "CustomerID": "Int64",
    },
    parse_dates=["InvoiceDate"],
)

print(f"Rows: {df_raw.shape[0]:,}")
print(f"Columns: {df_raw.shape[1]}")

Rows: 541,909
Columns: 8


In [4]:
df_raw.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850,United Kingdom


## Customer identifier quality investigation

Customer identifiers are required for customer-level analysis and RFM
segmentation. Before using them, we need to check for missing values, unusual
frequencies, and identifiers associated with implausibly diverse activity.

In [5]:
customer_id_quality = pd.Series(
    {
        "Total rows": len(df_raw),
        "Missing CustomerID values": df_raw["CustomerID"].isna().sum(),
        "Unique CustomerID values": df_raw["CustomerID"].nunique(),
    }
)

customer_id_quality

Total rows                   541909
Missing CustomerID values         0
Unique CustomerID values       4372
dtype: int64

In [7]:
customer_frequency = (
    df_raw["CustomerID"]
    .value_counts()
    .rename_axis("CustomerID")
    .reset_index(name="RowCount")
)

customer_frequency.head(10)

,CustomerID,RowCount
0,15287,135101
1,17841,7983
2,14911,5903
3,14096,5128
4,12748,4642
5,14606,2782
6,15311,2491
7,14646,2085
8,13089,1857
9,13263,1677


In [8]:
most_frequent_count = customer_frequency.loc[0, "RowCount"]
second_most_frequent_count = customer_frequency.loc[1, "RowCount"]

frequency_comparison = pd.Series(
    {
        "Most frequent CustomerID": customer_frequency.loc[0, "CustomerID"],
        "Most frequent row count": most_frequent_count,
        "Second-highest row count": second_most_frequent_count,
        "Ratio to second-highest": (
            most_frequent_count / second_most_frequent_count
        ),
        "Percentage of all rows": (
            most_frequent_count / len(df_raw) * 100
        ),
    }
)

frequency_comparison

Most frequent CustomerID    15,287.00
Most frequent row count    135,101.00
Second-highest row count     7,983.00
Ratio to second-highest         16.92
Percentage of all rows          24.93
dtype: float64

In [9]:
top_customers = customer_frequency.head(10).copy()
top_customers["CustomerID"] = top_customers["CustomerID"].astype(str)

fig = px.bar(
    top_customers,
    x="CustomerID",
    y="RowCount",
    title="Top 10 Customer Identifiers by Transaction Row Count",
    labels={
        "CustomerID": "Customer identifier",
        "RowCount": "Number of transaction rows",
    },
    color="RowCount",
    color_continuous_scale="Blues",
    text_auto=",",
)

fig.update_layout(
    xaxis_type="category",
    coloraxis_showscale=False,
)

fig.show()

In [10]:
suspected_customer_id = 15287

customer_15287 = df_raw.loc[
    df_raw["CustomerID"].eq(suspected_customer_id)
].copy()

customer_15287_summary = pd.Series(
    {
        "Transaction rows": len(customer_15287),
        "Percentage of all rows": (
            len(customer_15287) / len(df_raw) * 100
        ),
        "Unique invoices": customer_15287["InvoiceNo"].nunique(),
        "Unique products": customer_15287["StockCode"].nunique(),
        "Unique countries": customer_15287["Country"].nunique(),
        "First transaction": customer_15287["InvoiceDate"].min(),
        "Last transaction": customer_15287["InvoiceDate"].max(),
    }
)

customer_15287_summary

Transaction rows                       135101
Percentage of all rows                  24.93
Unique invoices                          3713
Unique products                          3811
Unique countries                            9
First transaction         2010-12-01 11:52:00
Last transaction          2011-12-09 10:26:00
dtype: object

In [11]:
customer_15287_countries = (
    customer_15287.groupby("Country", as_index=False)
    .size()
    .rename(columns={"size": "RowCount"})
    .sort_values("RowCount", ascending=False)
)

customer_15287_countries

,Country,RowCount
7,United Kingdom,133621
1,EIRE,711
3,Hong Kong,288
8,Unspecified,202
6,Switzerland,125
2,France,66
4,Israel,47
5,Portugal,39
0,Bahrain,2


In [12]:
fig = px.bar(
    customer_15287_countries,
    x="Country",
    y="RowCount",
    title="Country Distribution for Customer Identifier 15287",
    labels={
        "Country": "Country",
        "RowCount": "Number of transaction rows",
    },
    text_auto=",",
)

fig.update_layout(xaxis={"categoryorder": "total descending"})

fig.show()

### Possible median imputation

`CustomerID` is an identifier, not a numerical measurement, so its median has no customer-behaviour meaning. However, calculating it can help investigate whether a preprocessing step replaced missing identifiers with the median identifier value.

In [13]:
customer_id_median = df_raw["CustomerID"].median()

print(f"Median CustomerID: {customer_id_median:,.0f}")
print(
    "Does the most frequent identifier equal the median?",
    customer_id_median == suspected_customer_id,
)

Median CustomerID: 15,287
Does the most frequent identifier equal the median? True


## Raw-data integrity conclusion

The raw DataFrame is stored as `df_raw`. All cleaning and feature engineering will be performed on a separate copy so that the original imported data remains available for comparison.

The customer identifier investigation found that `15287` occurs in 135,101 rows, representing approximately 24.93% of the dataset. It is nearly 17 times more frequent than the next customer identifier and is associated with 3,713 invoices, 3,811 products, nine countries, and activity spanning almost the entire dataset period.

The value `15287` is also the median customer identifier. Because customer identifiers are categorical labels rather than meaningful numerical
measurements, this combination of findings strongly suggests that the supplied CSV may have replaced unknown customer identifiers with the median identifier.

This is an inference because the supplied CSV does not include a preprocessing record that confirms how the value was created.

The affected transaction rows will remain available for overall sales, product, time, and geographic analysis because their transaction information remains useful. However, identifier `15287` will be excluded from customer-level RFM segmentation because treating all of these rows as one customer would produce a misleading customer profile.

This approach may also exclude some genuine transactions belonging to the real customer `15287`. That trade-off will be documented as a project limitation.

## Dataset structure and quality assessment

Before transforming the dataset, its schema, missing values, duplicate records, cancellations, quantities, prices, and time coverage will be assessed.

The checks in this section do not change the raw DataFrame.

In [14]:
schema_summary = pd.DataFrame(
    {
        "Column": df_raw.columns,
        "DataType": df_raw.dtypes.astype(str).values,
        "NonNullCount": df_raw.notna().sum().values,
        "MissingCount": df_raw.isna().sum().values,
        "UniqueValues": df_raw.nunique(dropna=True).values,
    }
)

schema_summary

,Column,DataType,NonNullCount,MissingCount,UniqueValues
0,InvoiceNo,string,541909,0,25900
1,StockCode,string,541909,0,4070
2,Description,object,540455,1454,4223
3,Quantity,int64,541909,0,722
4,InvoiceDate,datetime64[ns],541909,0,23260
5,UnitPrice,float64,541909,0,1630
6,CustomerID,Int64,541909,0,4372
7,Country,object,541909,0,38


The dataset contains eight columns describing invoices, products, transaction dates, quantities, prices, customers, and countries.

`InvoiceNo`, `StockCode`, and `CustomerID` are identifiers rather than numerical measurements. `InvoiceDate` has been parsed as a datetime field. `Quantity` and `UnitPrice` are the principal numerical variables.

In [15]:
missing_summary = (
    df_raw.isna()
    .sum()
    .rename("MissingCount")
    .reset_index()
    .rename(columns={"index": "Column"})
)

missing_summary["MissingPercentage"] = (
    missing_summary["MissingCount"] / len(df_raw) * 100
)

missing_summary

,Column,MissingCount,MissingPercentage
0,InvoiceNo,0,0.00
1,StockCode,0,0.00
2,Description,1454,0.27
3,Quantity,0,0.00
4,InvoiceDate,0,0.00
5,UnitPrice,0,0.00
6,CustomerID,0,0.00
7,Country,0,0.00


In [16]:
fig = px.bar(
    missing_summary,
    x="Column",
    y="MissingCount",
    title="Missing Values by Dataset Column",
    labels={
        "Column": "Dataset column",
        "MissingCount": "Number of missing values",
    },
    text_auto=",",
)

fig.update_layout(xaxis_type="category")

fig.show()

Only `Description` contains explicitly missing values. However, the earlier customer investigation suggests that missing customer identifiers may have been replaced with `15287`, meaning that a missing-value count alone is not sufficient to assess customer data quality.

In [17]:
duplicate_count = df_raw.duplicated().sum()
duplicate_percentage = duplicate_count / len(df_raw) * 100

duplicate_summary = pd.Series(
    {
        "Exact duplicate rows": duplicate_count,
        "Duplicate percentage": duplicate_percentage,
    }
)

duplicate_summary

Exact duplicate rows   5,268.00
Duplicate percentage       0.97
dtype: float64

The dataset contains 5,268 exact duplicate rows. These records will be removed from the processed dataset to avoid counting the same transaction line more than once.

The raw dataset will remain unchanged.

In [18]:
cancellation_mask = (
    df_raw["InvoiceNo"]
    .str.upper()
    .str.startswith("C", na=False)
)

negative_quantity_mask = df_raw["Quantity"].lt(0)
zero_quantity_mask = df_raw["Quantity"].eq(0)
negative_price_mask = df_raw["UnitPrice"].lt(0)
zero_price_mask = df_raw["UnitPrice"].eq(0)

quality_summary = pd.DataFrame(
    {
        "Issue": [
            "Missing descriptions",
            "Exact duplicate rows",
            "Cancellation rows",
            "Negative quantities",
            "Zero quantities",
            "Negative unit prices",
            "Zero unit prices",
            "Suspected customer ID 15287",
        ],
        "RowCount": [
            df_raw["Description"].isna().sum(),
            df_raw.duplicated().sum(),
            cancellation_mask.sum(),
            negative_quantity_mask.sum(),
            zero_quantity_mask.sum(),
            negative_price_mask.sum(),
            zero_price_mask.sum(),
            df_raw["CustomerID"].eq(15287).sum(),
        ],
    }
)

quality_summary["PercentageOfDataset"] = (
    quality_summary["RowCount"] / len(df_raw) * 100
)

quality_summary

,Issue,RowCount,PercentageOfDataset
0,Missing descriptions,1454,0.27
1,Exact duplicate rows,5268,0.97
2,Cancellation rows,9288,1.71
3,Negative quantities,10624,1.96
4,Zero quantities,0,0.00
5,Negative unit prices,2,0.00
6,Zero unit prices,2515,0.46
7,Suspected customer ID 15287,135101,24.93


In [19]:
fig = px.bar(
    quality_summary,
    x="Issue",
    y="RowCount",
    title="Summary of Identified Data-Quality Conditions",
    labels={
        "Issue": "Data-quality condition",
        "RowCount": "Number of affected rows",
    },
    text_auto=",",
)

fig.update_layout(
    xaxis={
        "categoryorder": "total descending",
        "tickangle": -30,
    }
)

fig.show()

The conditions in the chart are not mutually exclusive. For example, a cancellation row may also contain a negative quantity and may also be an exact duplicate.

In [20]:
cancellation_summary = pd.Series(
    {
        "Cancellation rows": cancellation_mask.sum(),
        "Unique cancelled invoices": (
            df_raw.loc[cancellation_mask, "InvoiceNo"].nunique()
        ),
        "Negative-quantity rows": negative_quantity_mask.sum(),
        "Negative quantities without a C invoice": (
            negative_quantity_mask & ~cancellation_mask
        ).sum(),
    }
)

cancellation_summary

Cancellation rows                           9288
Unique cancelled invoices                   3836
Negative-quantity rows                     10624
Negative quantities without a C invoice     1336
dtype: int64

Cancellation invoices account for 9,288 rows. Negative quantities occur more frequently than invoice numbers beginning with `C`, showing that not every negative transaction line uses the cancellation invoice convention.

Cancellations and other negative-quantity rows will be retained in a separate returns and adjustments view. They will not be counted as completed sales.

In [21]:
df_raw.loc[
    negative_price_mask,
    [
        "InvoiceNo",
        "StockCode",
        "Description",
        "Quantity",
        "UnitPrice",
        "CustomerID",
        "Country",
    ],
]

,InvoiceNo,StockCode,Description,Quantity,UnitPrice,CustomerID,Country
299983,A563186,B,Adjust bad debt,1,"-11,062.06",15287,United Kingdom
299984,A563187,B,Adjust bad debt,1,"-11,062.06",15287,United Kingdom


The two negative-price records are bad-debt accounting adjustments rather than normal product sales. They will be excluded from completed product-sales calculations but retained in the quality and adjustment analysis.

Rows with zero unit prices cannot contribute sales revenue and will also be excluded from completed sales.

In [22]:
df_raw[["Quantity", "UnitPrice"]].describe(
    percentiles=[0.01, 0.25, 0.50, 0.75, 0.99]
).T

,count,mean,std,min,1%,25%,50%,75%,99%,max
Quantity,"541,909.00",9.55,218.08,"-80,995.00",-2.00,1.00,3.00,10.00,100.00,"80,995.00"
UnitPrice,"541,909.00",4.61,96.76,"-11,062.06",0.19,1.25,2.08,4.13,18.00,"38,970.00"


Quantity and unit price contain negative values, zero prices, and extreme outliers. These distributions are not expected to be normally distributed.

Outliers will not be removed solely because they are large. Their business meaning will be investigated, and robust transformations will be considered for machine learning.

In [23]:
coverage_summary = pd.Series(
    {
        "First transaction": df_raw["InvoiceDate"].min(),
        "Last transaction": df_raw["InvoiceDate"].max(),
        "Unique invoices": df_raw["InvoiceNo"].nunique(),
        "Unique products": df_raw["StockCode"].nunique(),
        "Unique customer identifiers": df_raw["CustomerID"].nunique(),
        "Unique countries": df_raw["Country"].nunique(),
    }
)

coverage_summary

First transaction              2010-12-01 08:26:00
Last transaction               2011-12-09 12:50:00
Unique invoices                              25900
Unique products                               4070
Unique customer identifiers                   4372
Unique countries                                38
dtype: object

## Data-quality assessment conclusion

The dataset contains 541,909 transaction rows covering December 2010 to December 2011.

The assessment identified:

- 1,454 missing product descriptions;
- 5,268 exact duplicate rows;
- 9,288 cancellation rows;
- 10,624 negative-quantity rows;
- 2,515 zero-price rows;
- two negative-price bad-debt adjustments;
- an unusually frequent customer identifier, `15287`.

The cleaning stage will create separate analytical datasets:

1. A completed-sales dataset containing positive quantities, positive prices, and non-cancellation invoices.
2. A returns-and-adjustments dataset containing cancellations, negative quantities, and accounting adjustments.
3. A customer-analysis dataset that excludes the unreliable identifier `15287`.

Exact duplicate rows will be removed from processed outputs. The original CSV and `df_raw` will remain unchanged.